# 02 · Stage 1: the assignment QUBO and the formulation check

**What it does.** Builds the QUBO that assigns each customer to a vehicle (`build_bqm`), then enumerates all
2⁸ = 256 bitstrings (`verify_formulation`) to prove that the lowest-energy bitstring is the feasible optimum.

**Why it was implemented.** TQ1 asks for a correct QUBO formulation. A *QUBO* is a loss function over 0/1
variables: $E(x) = \sum_i a_i x_i + \sum_{i<j} b_{ij} x_i x_j + c$. Constraints become *penalty* terms, which
act like a regulariser: zero when the rule holds, positive when it is broken. A full VRP QUBO would need
about 30 qubits, so the problem is decomposed ("cluster first, route second"). Stage 1 only decides
*which van serves which customer*:

$$E_1 = \sum_v \sum_{i<j} d_{ij} x_{i,v} x_{j,v} + P_1 \sum_i \Big(\sum_v x_{i,v} - 1\Big)^2 + P_2 \sum_v \Big(\sum_i x_{i,v} - 2.5\Big)^2$$

- Cost: distance between customers that share a van (close customers should ride together).
- One van per customer (penalty $P_1$).
- Capacity 3 **without slack qubits**: with 5 customers and 2 vans each van must take 2 or 3, and the target
  2.5 penalises loads 2 and 3 equally (0.25 each) and every other load more.
- Customer 1 is fixed to van 0 (relabelling the vans gives the same plan): 10 → **8 variables**.

The brute-force check is mandatory before any QAOA run. It verifies the formulation; it is not a benchmark.

**Resources.**
- Code: `domains/vrp/problem.py` (`build_bqm`, `build_qubo`, `decode`, `is_feasible`, `cluster_cost`, `penalty_bound`), `common/qubo_utils.py` (`verify_formulation`, `bqm_to_qp`).
- Libraries: dimod 0.12 (`BinaryQuadraticModel`, `add_linear_equality_constraint`, `ExactSolver`), qiskit-optimization 0.7 (`QuadraticProgram`).
- Derivation: `domains/vrp/README.md`, Stage 1 (coefficient tables; `tests/test_vrp_problem.py` checks every coefficient).
- Literature: A. Lucas, *Ising formulations of many NP problems*, Front. Phys. 2014; M. L. Fisher, R. Jaikumar, *A generalized assignment heuristic for vehicle routing*, Networks 1981.

In [ ]:
# Run from anywhere: move to the repository root so `common/`, `domains/`, `data/` resolve.
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
print("repository root:", ROOT)

In [ ]:
from common.qubo_utils import verify_formulation
from domains.vrp.problem import build_bqm, build_qubo, cluster_cost, decode, is_feasible, load_instance, penalty_bound

inst = load_instance("data/vrp/P-n6_1.vrp")
P = penalty_bound(inst)
penalties = {"one_vehicle": P, "capacity": P}
bqm = build_bqm(inst, penalties)
print("penalty weight P1 = P2 =", P)
print("variables:", list(bqm.variables))
print("couplings:", bqm.num_interactions, "| constant offset:", bqm.offset)

## Penalty weight
`penalty_bound` = the largest total distance from one customer to all others (customer 4: 51 + 43 + 21 + 47 = 162).
Moving, dropping or duplicating one customer changes the cost by at most that much, so with P = 162 a single
violation never pays. The brute-force check below confirms the full formulation.

## The QUBO coefficients
After expanding the squares ($x^2 = x$ for binary $x$) and fixing customer 1, each term has a closed form
(`domains/vrp/README.md`, Stage 1 step 5): e.g. $x_{j,1}$ has $-P_1 - 4P_2 = -810$, and $x_{j,0}x_{j,1}$ has $2P_1 = 324$.

In [ ]:
print("linear terms:")
for v, a in bqm.linear.items():
    print(f"  {v}: {a:g}")
print("quadratic terms:")
for (u, v), b in bqm.quadratic.items():
    print(f"  {u} * {v}: {b:g}")

The same model as a Qiskit `QuadraticProgram` (what QAOA receives):

In [ ]:
print(build_qubo(inst, penalties).prettyprint())

## Brute-force formulation check
`verify_formulation` scores all 256 bitstrings and raises an error if any lowest-energy bitstring is infeasible.
The **gap** is the energy distance from the best feasible to the cheapest infeasible bitstring.

In [ ]:
feasible = lambda s: is_feasible(decode(s, inst))
report = verify_formulation(bqm, feasible)
for key in ("n_vars", "n_states", "n_feasible", "ground_energy", "best_infeasible_energy", "gap"):
    print(f"{key}: {report[key]}")
ground = decode(report["ground_states"][0], inst)
print("lowest-energy assignment:", ground.routes, "| cluster cost:", cluster_cost(ground))

## All feasible assignments
For every feasible bitstring the penalties are constant ($0.5 P_2 = 81$), so energy = cluster cost + 81.
The last column checks the decomposition: the best route length that each split allows (found by trying every
visit order). The split Stage 1 prefers is also the one with the shortest routes.

In [ ]:
import dimod
from itertools import permutations
from domains.vrp.problem import Solution, objective

def best_route_length(group):
    return min(objective(Solution(inst, (order,))) for order in permutations(group))

rows = []
for s, e in dimod.ExactSolver().sample(bqm).data(["sample", "energy"]):
    sol = decode(s, inst)
    if is_feasible(sol):
        rows.append((e, sol.routes, cluster_cost(sol), sum(best_route_length(g) for g in sol.routes)))
print(f"{'van 0':<12}{'van 1':<12}{'cluster cost':>13}{'energy':>9}{'best routes':>13}")
for e, (v0, v1), cc, length in sorted(rows):
    print(f"{str(v0):<12}{str(v1):<12}{cc:>13g}{e:>9g}{length:>13g}")

## How small can the penalty be?
The analytic bound is sufficient, not tight. Below some weight, breaking a rule becomes cheaper than following
it and the check fails. Large penalties also matter for QAOA (TQ3, notebook 07).

In [ ]:
def passes(p):
    try:
        verify_formulation(build_bqm(inst, {"one_vehicle": p, "capacity": p}), feasible)
        return True
    except ValueError:
        return False

threshold = {p: passes(p) for p in (10, 20, 30, 40, 60, 100, 162)}
for p, ok in threshold.items():
    print(f"P = {p:>3}: {'passes' if ok else 'FAILS: an infeasible bitstring has the lowest energy'}")

## Expected results
- P = 162; **8 variables**, 16 couplings, constant offset 2025.
- 10 feasible bitstrings out of 256.
- Unique lowest-energy assignment: van 0 = {1, 2, 5}, van 1 = {3, 4}; energy **134** = cluster cost 53 + 81; gap **140**.
- That split also has the shortest routes (118), the next split by cluster cost (98) needs 131.
- The check fails at P = 20 and passes from P = 40.

In [ ]:
assert P == 162 and bqm.num_variables == 8 and bqm.num_interactions == 16 and bqm.offset == 2025
assert report["n_feasible"] == 10 and report["ground_energy"] == 134 and report["gap"] == 140
assert ground.routes == ((1, 2, 5), (3, 4)) and cluster_cost(ground) == 53
assert min(rows)[3] == 118
assert not threshold[20] and threshold[40]